In [2]:
import pandas as pd
import numpy as np
import altair

# Crear els Dataframes del XML

In [37]:
location_columns = ["numeroparte", "idcomunidad", "idprovincia",
                    "idmunicipio", "nummunicipiosafectados", "latitud", "longitud"]

locations = pd.read_xml("dades/Xml_20261116_123725_1.xml", xpath="./Pif/pif_localizacion")[location_columns]
locations.head()

,numeroparte,idcomunidad,idprovincia,idmunicipio,nummunicipiosafectados,latitud,longitud
0,1968080028,2,8,0,1,NaN,NaN
1,1968080029,2,8,0,1,NaN,NaN
2,1968080030,2,8,0,1,NaN,NaN
3,1968080031,2,8,0,1,NaN,NaN
4,1968080032,2,8,0,1,NaN,NaN


In [44]:
municipis_dades = pd.read_excel("dades/diccionario26.xlsx", header=1)
municipis_dades = municipis_dades.loc[municipis_dades["CODAUTO"]==9, ["CPRO", "CMUN", "DC", "NOMBRE"]]
print(municipis_dades.loc[municipis_dades["CPRO"]==8])
municipis_dades.head()

      CPRO  CMUN  DC                NOMBRE
862      8     1   8                Abrera
863      8     2   3    Aguilar de Segarra
864      8    14   2            Aiguafreda
865      8     3   9                Alella
866      8     4   4                Alpens
...    ...   ...  ..                   ...
1168     8   307   3  Vilanova i la Geltrú
1169     8   214   0      Vilassar de Dalt
1170     8   219   1       Vilassar de Mar
1171     8   304   1    Vilobí del Penedès
1172     8   308   9     Viver i Serrateix

[311 rows x 4 columns]


,CPRO,CMUN,DC,NOMBRE
862,8,1,8,Abrera
863,8,2,3,Aguilar de Segarra
864,8,14,2,Aiguafreda
865,8,3,9,Alella
866,8,4,4,Alpens


In [57]:
provincies = ["08", "17", "25", "43"]

path = "dades/MUNICIPIOS.csv"

municipis_coordenades = pd.read_csv(
    path,
    encoding="cp1252",
    sep=";",
    dtype="string",
)

municipis_coordenades = municipis_coordenades.loc[municipis_coordenades["COD_PROV"].isin(provincies)]
municipis_coordenades.info()
municipis_coordenades.head()

<class 'pandas.core.frame.DataFrame'>
Index: 947 entries, 862 to 6544
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   COD_INE                   947 non-null    string
 1   ID_REL                    947 non-null    string
 2   COD_GEO                   947 non-null    string
 3   COD_PROV                  947 non-null    string
 4   PROVINCIA                 947 non-null    string
 5   NOMBRE_ACTUAL             947 non-null    string
 6   POBLACION_MUNI            947 non-null    string
 7   SUPERFICIE                947 non-null    string
 8   PERIMETRO                 947 non-null    string
 9   COD_INE_CAPITAL           947 non-null    string
 10  CAPITAL                   947 non-null    string
 11  POBLACION_CAPITAL         947 non-null    string
 12  HOJA_MTN25                947 non-null    string
 13  LONGITUD_ETRS89_REGCAN95  947 non-null    string
 14  LATITUD_ETRS89_REGCAN95   94

,COD_INE,ID_REL,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,POBLACION_CAPITAL,HOJA_MTN25,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,ORIGENCOOR,ALTITUD,ORIGENALTITUD
862,08001000000,1080018,08003,08,Barcelona,Abrera,13207,"1999,03",24837,08001001401,Abrera,9579,0392-3,"1,901568685","41,5163976",Detección automática,106,MDT
863,08002000000,1080023,08006,08,Barcelona,Aguilar de Segarra,293,"4313,9082",39627,08002000101,Aguilar de Segarra,62,0362-3,"1,631134116","41,73926837",Investigación,490,Investigación
864,08003000000,1080039,08009,08,Barcelona,Alella,10251,"970,22",14368,08003000101,Alella,2382,0421-1,"2,295316458","41,49361157",Investigación,90,Investigación
865,08004000000,1080044,08012,08,Barcelona,Alpens,267,"1367,7",23029,08004000101,Alpens,259,0293-2,"2,101098216","42,11928971",Detección automática,857,MDT
866,08005000000,1080057,08015,08,Barcelona,L'Ametlla del Vallès,9473,"1424,81",21836,08005000101,L'Ametlla del Vallès,4412,0364-3,"2,261438098","41,66955386",Detección automática,290,MDT


In [58]:
def municipality_code(province, municipality):
    return (
        province.astype("Int64").astype("string").str.zfill(2)
        + municipality.astype("Int64").astype("string").str.zfill(3)
    )


def parse_coordinate(values):
    text = values.astype("string").str.strip().str.replace(",", ".", regex=False)
    return pd.to_numeric(text, errors="coerce")


fires = locations.assign(
    codi_municipi=municipality_code(locations["idprovincia"], locations["idmunicipio"]),
    latitud=parse_coordinate(locations["latitud"]),
    longitud=parse_coordinate(locations["longitud"]),
)

towns = municipis_dades.assign(
    codi_municipi=municipality_code(municipis_dades["CPRO"], municipis_dades["CMUN"]),
    municipi=municipis_dades["NOMBRE"].str.strip(),
)[["codi_municipi", "municipi"]]

coords = municipis_coordenades.assign(
    codi_municipi=municipis_coordenades["COD_INE"].str.strip().str[:5],
    latitud_municipi=parse_coordinate(municipis_coordenades["LATITUD_ETRS89_REGCAN95"]),
    longitud_municipi=parse_coordinate(municipis_coordenades["LONGITUD_ETRS89_REGCAN95"]),
)[["codi_municipi", "latitud_municipi", "longitud_municipi"]]

provinces = (
    municipis_coordenades[["COD_PROV", "PROVINCIA"]]
    .rename(columns={"COD_PROV": "idprovincia", "PROVINCIA": "provincia"})
    .assign(
        idprovincia=lambda df: df["idprovincia"].astype("Int64"),
        provincia=lambda df: df["provincia"].str.strip(),
    )
    .dropna(subset=["idprovincia"])
    .drop_duplicates()
)

merged = (
    fires.merge(towns.dropna(subset=["codi_municipi"]), on="codi_municipi",
                how="left", validate="many_to_one")
    .merge(coords.dropna(subset=["codi_municipi"]), on="codi_municipi",
           how="left", validate="many_to_one")
    .merge(provinces, on="idprovincia", how="left", validate="many_to_one")
)

merged["latitud"] = merged["latitud"].fillna(merged["latitud_municipi"])
merged["longitud"] = merged["longitud"].fillna(merged["longitud_municipi"])

locations_enriched = merged[[
    "idprovincia",
    "idmunicipio",
    "municipi",
    "provincia",
    "latitud",
    "longitud",
    "nummunicipiosafectados",
]]

In [66]:
url = "https://idescat.cat/codis/?id=50&n=84"

comarques = pd.read_html(
    url,
    match="Codi municipi",
    converters={"Codi municipi": str},
)[0][["Codi municipi", "Nom comarca"]].copy()

comarques["codi_municipi"] = comarques["Codi municipi"].str.strip().str[:5]

comarca_by_municipality = comarques.set_index(
    "codi_municipi", verify_integrity=True
)["Nom comarca"]

locations_enriched = locations_enriched.assign(
    comarca=merged["codi_municipi"].map(comarca_by_municipality)
)

In [67]:
locations_enriched.info()
locations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31319 entries, 0 to 31318
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   idprovincia             31319 non-null  int64  
 1   idmunicipio             31319 non-null  int64  
 2   municipi                24858 non-null  object 
 3   provincia               31319 non-null  string 
 4   latitud                 24873 non-null  Float64
 5   longitud                24873 non-null  Float64
 6   nummunicipiosafectados  31319 non-null  int64  
 7   comarca                 24858 non-null  object 
dtypes: Float64(2), int64(3), object(2), string(1)
memory usage: 2.0+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31319 entries, 0 to 31318
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   numeroparte             31319 non-null  int64  
 1   idcomunidad      

In [68]:
locations_enriched.loc[locations_enriched["municipi"].notna()].head()

,idprovincia,idmunicipio,municipi,provincia,latitud,longitud,nummunicipiosafectados,comarca
6436,8,284,Tordera,Barcelona,41.70172,2.718588,1,Maresme
6437,8,253,Santa Maria de Besora,Barcelona,42.126762,2.258205,1,Osona
6438,8,125,Montcada i Reixac,Barcelona,41.480027,2.18722,1,Vallès Occidental
6439,8,111,Malla,Barcelona,41.894606,2.243543,1,Osona
6440,8,82,Fogars de la Selva,Barcelona,41.735128,2.671402,1,Selva


# Bla bla bla

In [10]:
path = "dades/Xml_20261116_123725_1.xml"

sections = {
    name: pd.read_xml(path, xpath=f"./Pif/pif_{name}")
    for name in ("localizacion", "tiempos", "causa")
}

for name, frame in sections.items():
    print(name, frame.columns.tolist())

localizacion ['numeroparte', 'idcomunidad', 'idprovincia', 'idmunicipio', 'nummunicipiosafectados', 'hoja', 'cuadricula', 'idcomarcaisla', 'paraje', 'huso', 'x', 'y', 'latitud', 'longitud', 'iddatum', 'puntosinicioincendio', 'identidadmenor']
tiempos ['numeroparte', 'deteccion', 'llegadapmt', 'llegadapmae', 'llegadapbh', 'llegadapac', 'controlado', 'extinguido']
causa ['numeroparte', 'idcausa', 'idcertidumbrecausa', 'idcausante', 'idclasedia', 'idmotivacion', 'diastormenta', 'idgradoresponsabilidad', 'idinvestigacioncausa', 'idautorizacionactividad', 'motivacionotros', 'causaotros']


In [11]:
sections["localizacion"].head()

,numeroparte,idcomunidad,idprovincia,idmunicipio,nummunicipiosafectados,hoja,cuadricula,idcomarcaisla,paraje,huso,x,y,latitud,longitud,iddatum,puntosinicioincendio,identidadmenor
0,1968080028,2,8,0,1,NaN,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1968080029,2,8,0,1,NaN,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1968080030,2,8,0,1,NaN,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1968080031,2,8,0,1,NaN,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1968080032,2,8,0,1,NaN,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
